# 02 · SFT：把续写模型变成指令助手

**本节目标**

1. 理解 SFT 数据中的 `user` / `assistant` 消息和对话模板。
2. 看懂标签掩码：只在助手回复位置计算监督损失。
3. 接着第一节的 Pretrain 权重，在一张 24 GB GPU 上用 256 条样本跑通 Full SFT，并检查新权重。

首次下载 `sft_t2t_mini.jsonl` 约 1.7 GB；短实验只取其中 256 条。短训练用于验证流程，不能代表真实指令能力。源码：[MiniMind](https://github.com/jingyaogong/minimind)；数据：[ModelScope MiniMind 数据集](https://modelscope.cn/datasets/gongjy/minimind_dataset)。

## 1. 定位课程、确认 GPU

先运行 [第一节 Pretrain](01_pretrain.ipynb)，确认 `work/runs/pretrain/course_pretrain_768.pth` 已存在。Notebook 启动时如果工作目录与仓库目录不同，可手动设置 `COURSE_ROOT`。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import torch

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
candidates += [p.parent.parent for p in Path.cwd().glob("*/notebooks/02_sft.ipynb")]
candidates += [p.parent.parent for p in Path.cwd().glob("*/*/notebooks/02_sft.ipynb")]
COURSE_ROOT = next((p for p in candidates if (p / "notebooks" / "02_sft.ipynb").is_file()), None)
if COURSE_ROOT is None:
    raise FileNotFoundError("请将 COURSE_ROOT 手动设为课程仓库根目录")
WORK = COURSE_ROOT / "work"
PRETRAIN_WEIGHT = WORK / "runs" / "pretrain" / "course_pretrain_768.pth"
assert PRETRAIN_WEIGHT.is_file(), "先完成 01_pretrain.ipynb，生成课程预训练权重"
assert torch.cuda.is_available(), "请启动 CUDA GPU Notebook 实例"
print("课程目录:", COURSE_ROOT)
print("GPU:", torch.cuda.get_device_name(0))
print("显存 GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("预训练权重:", PRETRAIN_WEIGHT)

课程目录: /home/yang/data2/workspace/minimind-notebook-course
GPU: NVIDIA A100-SXM4-80GB
显存 GiB: 79.3
预训练权重: /home/yang/data2/workspace/minimind-notebook-course/work/runs/pretrain/course_pretrain_768.pth


## 2. 准备上游源码和 SFT 数据

沿用第一节克隆的 MiniMind 源码。SFT 数据每行的 `conversations` 是消息列表；完整数据可能包含多轮对话、思维内容和工具调用，本节只选取普通的第一组用户提问与助手回答，方便观察基础监督学习。

In [2]:
required = ("modelscope", "datasets", "transformers", "sentencepiece")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"缺少 {missing}；在新单元运行 %pip install modelscope datasets transformers sentencepiece，重启内核后再运行")
from modelscope.hub.file_download import dataset_file_download
from transformers import AutoTokenizer

SOURCE = WORK / "minimind"
if not (SOURCE / "trainer" / "train_full_sft.py").is_file():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jingyaogong/minimind.git", str(SOURCE)], check=True)
filename = "sft_t2t_mini.jsonl"
nearby = COURSE_ROOT.parent / "minimind_dataset" / filename
cached = WORK / "datasets" / filename
if nearby.is_file():
    DATA_FILE = nearby
elif cached.is_file():
    DATA_FILE = cached
else:
    cached.parent.mkdir(parents=True, exist_ok=True)
    DATA_FILE = Path(dataset_file_download(dataset_id="gongjy/minimind_dataset", file_path=filename, local_dir=str(cached.parent)))
assert DATA_FILE.is_file()
tokenizer = AutoTokenizer.from_pretrained(str(SOURCE / "model"))
print("源码:", SOURCE)
print("数据:", DATA_FILE, "大小 GiB:", round(DATA_FILE.stat().st_size / 2**30, 2))

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


源码: /home/yang/data2/workspace/minimind-notebook-course/work/minimind
数据: /home/yang/data2/workspace/minimind_dataset/sft_t2t_mini.jsonl 大小 GiB: 1.62


## 3. 观察对话模板和标签掩码

MiniMind 的 `SFTDataset` 会先用 tokenizer 的 chat template 拼接消息，再将非助手回复位置的 label 设为 `-100`。交叉熵忽略这些位置，所以用户提问提供上下文，训练目标是助手回复。

In [3]:
with DATA_FILE.open("r", encoding="utf-8") as f:
    example = json.loads(next(f))
messages = example["conversations"]
print("角色顺序:", [m["role"] for m in messages[:6]])
print("第一条消息:", messages[0].get("content", "")[:150])
print("模板片段:", tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=False)[:350])

角色顺序: ['user', 'assistant', 'user', 'assistant', 'user', 'assistant']
第一条消息: 你背后的模型是哪个版本？它由谁开发？
模板片段: <|im_start|>user
你背后的模型是哪个版本？它由谁开发？<|im_end|>
<|im_start|>assistant
<think>

</think>

我是由jingyaogong开发的高效小参数AI模型。<|im_end|>



## 4. 取 256 条短实验样本

选择第一组普通 `user` → `assistant` 对话，仅保留 `role` 和 `content`。用 tokenizer 过滤过长样本，减少截断导致的空监督。训练文件仍保持上游期望的 JSONL 结构。

In [4]:
SMALL_DATA = WORK / "datasets" / "sft_course_256.jsonl"
SMALL_DATA.parent.mkdir(parents=True, exist_ok=True)
selected = []
with DATA_FILE.open("r", encoding="utf-8") as src:
    for line in src:
        row = json.loads(line)
        conv = row.get("conversations", [])
        user = next((m for m in conv if m.get("role") == "user" and m.get("content")), None)
        if user is None:
            continue
        assistant = next((m for m in conv[conv.index(user)+1:] if m.get("role") == "assistant" and m.get("content")), None)
        if assistant is None:
            continue
        pair = [{"role": "user", "content": user["content"]}, {"role": "assistant", "content": assistant["content"]}]
        rendered = tokenizer.apply_chat_template(pair, tokenize=False, add_generation_prompt=False)
        if len(tokenizer(rendered).input_ids) > 260:
            continue
        selected.append({"conversations": pair})
        if len(selected) == 256:
            break
assert len(selected) >= 64, f"可用样本过少: {len(selected)}"
with SMALL_DATA.open("w", encoding="utf-8") as dst:
    for row in selected:
        dst.write(json.dumps(row, ensure_ascii=False) + "\n")
print("短实验样本:", len(selected), "路径:", SMALL_DATA)

短实验样本: 256 路径: /home/yang/data2/workspace/minimind-notebook-course/work/datasets/sft_course_256.jsonl


In [5]:
sys.path.insert(0, str(SOURCE))
from dataset.lm_dataset import SFTDataset
sft_ds = SFTDataset(str(SMALL_DATA), tokenizer, max_length=340)
input_ids, labels = sft_ds[0]
active = labels != -100
print("序列长度:", len(input_ids), "参与损失的 token:", int(active.sum()))
print("助手目标片段:", tokenizer.decode(labels[active][:80]))
assert active.any(), "首条样本没有有效的助手标签"

2026-11-25 11:39:36 - datasets - INFO - PyTorch version 2.11.0+cu128 available.


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 256 examples [00:00, 37343.64 examples/s]

序列长度: 340 参与损失的 token: 19
助手目标片段: 我是由jingyaogong开发的高效小参数AI模型。<|im_end|>



## 5. 从 Pretrain 权重接着训练

上游 `init_model` 默认在其源码目录的 `out/` 查找起点权重。下面创建仅供本课程使用的软链接，文件仍留在 `work/runs/pretrain/`。`--from_weight course_pretrain` 明确指定起点；SFT 输出到另一目录。有效 batch 为 `8 × 4 = 32`。

In [6]:
source_out = SOURCE / "out"
source_out.mkdir(parents=True, exist_ok=True)
weight_link = source_out / "course_pretrain_768.pth"
if weight_link.is_symlink():
    assert weight_link.resolve() == PRETRAIN_WEIGHT.resolve(), "已有同名链接指向其他文件"
elif weight_link.exists():
    raise FileExistsError(f"请检查已有文件，避免覆盖: {weight_link}")
else:
    weight_link.symlink_to(PRETRAIN_WEIGHT)
OUT = WORK / "runs" / "sft"
OUT.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, "train_full_sft.py",
    "--data_path", str(SMALL_DATA),
    "--save_dir", str(OUT),
    "--save_weight", "course_sft",
    "--from_weight", "course_pretrain",
    "--epochs", "1",
    "--batch_size", "8",
    "--accumulation_steps", "4",
    "--max_seq_len", "340",
    "--num_workers", "2",
    "--log_interval", "10",
    "--save_interval", "10000",
]
print("运行:", " ".join(cmd))

运行: /home/yang/data2/workspace/minimind-notebook-course/work/jupyter-venv/bin/python train_full_sft.py --data_path /home/yang/data2/workspace/minimind-notebook-course/work/datasets/sft_course_256.jsonl --save_dir /home/yang/data2/workspace/minimind-notebook-course/work/runs/sft --save_weight course_sft --from_weight course_pretrain --epochs 1 --batch_size 8 --accumulation_steps 4 --max_seq_len 340 --num_workers 2 --log_interval 10 --save_interval 10000


### 观察显存

若环境提供 `nvidia-smi`，训练期间每 0.5 秒读取一次当前 GPU 的整卡显存使用量，并打印峰值。单卡 Notebook 通常无需设置 GPU 编号；多卡服务器可在启动 Notebook 前设置 `CUDA_VISIBLE_DEVICES=6`。整卡读数包含同卡其他进程。

In [7]:
gpu_visible = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0]
gpu_index = gpu_visible if gpu_visible.isdigit() else "0"
readings_mib = []
stop_monitor = threading.Event()
def watch_gpu():
    if shutil.which("nvidia-smi") is None:
        return
    while not stop_monitor.is_set():
        result = subprocess.run(
            ["nvidia-smi", f"--id={gpu_index}", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            try:
                readings_mib.append(int(result.stdout.strip().splitlines()[0]))
            except (ValueError, IndexError):
                pass
        stop_monitor.wait(0.5)
monitor = threading.Thread(target=watch_gpu, daemon=True)
monitor.start()
try:
    subprocess.run(cmd, cwd=SOURCE / "trainer", check=True)
finally:
    stop_monitor.set()
    monitor.join(timeout=3)
if readings_mib:
    print("整卡显存峰值 MiB:", max(readings_mib), "约 GiB:", round(max(readings_mib) / 1024, 2))
else:
    print("无法从 nvidia-smi 读取显存；可在终端运行 nvidia-smi 观察")

2026-11-25 11:39:42 - datasets - INFO - PyTorch version 2.11.0+cu128 available.


Model Params: 63.91M
Trainable Params: 63.912M


Epoch:[1/1](10/32), loss: 7.5217, logits_loss: 7.5217, aux_loss: 0.0000, lr: 0.00000800, epoch_time: 0.0min
Epoch:[1/1](20/32), loss: 7.3124, logits_loss: 7.3124, aux_loss: 0.0000, lr: 0.00000378, epoch_time: 0.0min
Epoch:[1/1](30/32), loss: 7.3902, logits_loss: 7.3902, aux_loss: 0.0000, lr: 0.00000109, epoch_time: 0.0min
Epoch:[1/1](32/32), loss: 7.4566, logits_loss: 7.4566, aux_loss: 0.0000, lr: 0.00000100, epoch_time: 0.0min


整卡显存峰值 MiB: 3157 约 GiB: 3.08


## 6. 检查 SFT 权重

比较两阶段权重的文件大小与参数数目，并检查 SFT 权重的数值。权重生成成功说明训练链路已走通；想判断回复质量，还需要设计独立的推理和评测实验。

In [8]:
SFT_WEIGHT = OUT / "course_sft_768.pth"
assert SFT_WEIGHT.is_file(), f"未找到 SFT 权重: {SFT_WEIGHT}"
weights = torch.load(SFT_WEIGHT, map_location="cpu", weights_only=True)
parameter_count = sum(t.numel() for t in weights.values())
all_finite = all(torch.isfinite(t).all().item() for t in weights.values())
print("SFT 权重:", SFT_WEIGHT)
print("大小 MiB:", round(SFT_WEIGHT.stat().st_size / 2**20, 1))
print("参数量 M:", round(parameter_count / 1e6, 2))
print("参数均为有限数:", all_finite)
assert all_finite

SFT 权重: /home/yang/data2/workspace/minimind-notebook-course/work/runs/sft/course_sft_768.pth
大小 MiB: 131.3
参数量 M: 68.83
参数均为有限数: True


## 回顾与练习

- **Pretrain 与 SFT 的差别**：前者学习文本中的 next-token 规律；后者通过带角色的对话样本学习按指令回复。
- **为什么掩码用户文本？** 用户消息提供条件，loss 聚焦于助手目标文本。
- **练习**：把样本上限改成 512，记录 loss 和显存；把 `max_seq_len` 改成 512，比较显存变化。增加长度时应先降低 batch，避免超出显存。
- **完整训练**：使用全部 `sft_t2t_mini.jsonl`，参照 [MiniMind Full SFT 脚本](https://github.com/jingyaogong/minimind/blob/master/trainer/train_full_sft.py) 和数据许可；当前 256 条实验不能反映模型的实际聊天能力。